# ⚽ Data Preprocessing & Visualization: Zero to Master (Pro Edition)

Welcome to the Elite Training Center, Coach! Today we're moving from a simple kickabout to professional team management. We'll follow a strict 6-step workflow to prepare our squad for the World Cup.

### **The Elite Game Plan:**
1. **Load & Explore:** Scout the raw talent.
2. **Handle Missing Values:** Fix the medical records.
3. **Data Visualization:** Analyze the game tapes.
4. **Outlier Detection:** Manage the superstar egos (salaries).
5. **Duplicates & Variance:** Trim the squad depth.
6. **Engineering & Encoding:** Final tactics for the match.

## 🏟️ Pre-Match: Create the Messy Dataset

**Coach's Note:** Real-world scouting reports are messy. We'll generate a dataset with identifiers, missing values, outliers, duplicates, and low-variance features.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filter_type = 'ignore'

np.random.seed(42)
rows = 200

data = {
    'Player_ID': [f'ID_{1000+i}' for i in range(rows)],
    'Player_Name': [f'Player_{i}' for i in range(rows)],
    'Position': np.random.choice(['Striker', 'Midfielder', 'Defender', 'Goalkeeper', 'Bench'], rows, p=[0.2, 0.3, 0.3, 0.1, 0.1]),
    'Age': np.random.randint(18, 40, rows),
    'Weekly_Salary': np.concatenate([np.random.normal(5000, 1000, rows-10), [500000, 600000, 750000, 800000, 1000000, 1200000, 1500000, 2000000, 500, 600]]),
    'Goals_Scored': np.random.poisson(5, rows),
    'Market_Value': np.random.exponential(100000, rows),
    'Signing_Date': np.random.choice(['2023-01-01', '2023-06-15', '2024-02-10'], rows),
    'Medical_Clearance': np.random.choice(['Yes', 'No', np.nan], rows, p=[0.6, 0.1, 0.3]),
    'Fitness_Score': np.random.choice([np.nan, 85.0, 45.0], rows, p=[0.4, 0.5, 0.1]),
    'Always_Same_Stat': 10  # Low variance column
}

football_df = pd.DataFrame(data)
# Adding duplicates
football_df = pd.concat([football_df, football_df.iloc[:10]], ignore_index=True)
football_df.to_csv('raw_football_data.csv', index=False)
print("Raw scouting data saved!")

## 📋 Step 1: Load and Explore the Dataset

**Coach's Note:** First look at the squad list. We need to identify who is who and what kind of stats we have.

In [ ]:
# 1. Load
df = pd.read_csv('raw_football_data.csv')

# 2. Preview
print("--- First 5 Rows ---")
display(df.head())

# 3. Drop unnecessary columns (Identifiers)
df.drop(['Player_ID', 'Player_Name'], axis=1, inplace=True)

# 4. Check data types
print("\n--- Data Types ---")
print(df.dtypes)

# 5. Convert categorical columns to 'category' type
for col in ['Position', 'Medical_Clearance']:
    df[col] = df[col].astype('category')

# 6. Check unique categories
print("\n--- Unique Categories ---")
for col in df.select_dtypes('category').columns:
    print(f"{col}: {df[col].nunique()} unique categories")

## 🏥 Step 2: Handle Missing Values

**Coach's Note:** We can't have players with missing medical info! If a column is mostly empty, we cut it. Otherwise, we fill based on the stat's distribution.

In [ ]:
# 1. Check for missing values & calculate ratios
null_counts = df.isnull().sum()
null_ratios = null_counts / len(df)
print("--- Missing Value Ratios ---")
print(null_ratios)

# 2. Drop columns with > 50% nulls (Excessive missing entries)
threshold = 0.5
cols_to_drop = null_ratios[null_ratios > threshold].index.tolist()
df.drop(cols_to_drop, axis=1, inplace=True)
print(f"\nDropped columns: {cols_to_drop}")

# 3. Fill null values
# Categorical -> Mode
for col in df.select_dtypes(['category', 'object']).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Numeric -> Median (Robust to outliers in skewed data)
for col in df.select_dtypes(['int64', 'float64']).columns:
    if df[col].isnull().any():
        # Check skewness
        skew = df[col].skew()
        print(f"{col} Skewness: {skew:.2f}")
        df[col] = df[col].fillna(df[col].median())

# 4. Final Check
print("\nRemaining missing values:", df.isnull().sum().sum())

## 📊 Step 3: Data Visualization

### **What is Visualization?**
Visualization is the art of turning raw statistics into pictures. It helps coaches spot patterns, errors, and superstar performances that raw numbers can't show easily.

### **A. Data Distribution (The Squad Shape)**

In [ ]:
plt.figure(figsize=(15, 10))

# 1. Numeric: Histogram & KDE (Distribution of Age)
plt.subplot(2, 2, 1)
sns.histplot(df['Age'], kde=True, color='blue')
plt.title('Age Distribution')

# 2. Categorical: Count Plot (Positions Available)
plt.subplot(2, 2, 2)
sns.countplot(data=df, x='Position', palette='viridis')
plt.title('Squad Positions')

# 3. Categorical: Pie Plot (Clearance Status)
plt.subplot(2, 2, 3)
df['Medical_Clearance'].value_counts().plot.pie(autopct='%1.1f%%', colors=['green', 'red'])
plt.title('Medical Clearance %')

plt.tight_layout()
plt.show()

### **B. Outlier Detection (The Egos)**

In [ ]:
# Box Plot for Weekly Salary
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['Weekly_Salary'], color='salmon')
plt.title('Salary Box Plot (Spotting the Superstars)')
plt.show()

### **C. Relationship Graphs (Tactical Links)**

In [ ]:
# 1. Numerical/Numerical: Scatter Plot (Age vs Goals)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Age', y='Goals_Scored', hue='Position')
plt.title('Age vs Goals Scored')
plt.show()

# 2. Numerical/Categorical: Bar Plot (Avg Goals by Position)
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x='Position', y='Goals_Scored', estimator=np.mean)
plt.title('Average Goals per Position')
plt.show()

# 3. Categorical/Categorical: Heatmap (Position vs Clearance - CrossTab)
plt.figure(figsize=(8, 5))
ct = pd.crosstab(df['Position'], df['Medical_Clearance'])
sns.heatmap(ct, annot=True, cmap='YlGnBu')
plt.title('Relationship: Position vs Medical Clearance')
plt.show()

# 4. Correlation Matrix Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df.select_dtypes(['int64', 'float64']).corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

## 🥊 Step 4: Detect and Handle Outliers

**Coach's Note:** We've seen the salary boxplot. Some players earn way too much or too little. We'll cap them using the 'Whiskers' of our boxplot (IQR Rule).

In [ ]:
for col in ['Weekly_Salary', 'Market_Value']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_whisker = Q1 - 1.5 * IQR
    upper_whisker = Q3 + 1.5 * IQR
    
    # Capping
    df[col] = df[col].clip(lower=lower_whisker, upper=upper_whisker)

print("Outliers have been capped to whiskers.")
sns.boxplot(x=df['Weekly_Salary']) # Verification

### **Handling Rare Categories**
Infrequent categories cause confusion. Let's merge them into 'Other' or replace with Mode.

In [ ]:
pos_counts = df['Position'].value_counts()
rare_pos = pos_counts[pos_counts < 10].index.tolist()
print(f"Rare positions: {rare_pos}")

df['Position'] = df['Position'].apply(lambda x: 'Other' if x in rare_pos else x)
print("Rare categories merged.")

## 👯 Step 5: Check and Remove Duplicates

**Coach's Note:** We don't want the same player taking two spots on the bus!

In [ ]:
print(f"Duplicates found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Clean squad size: {len(df)}")

## 📉 Step 6: Feature Variance Check

**Coach's Note:** If a stat is the same for every player (like having a pulse), it doesn't help us pick a winner. Drop it!

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
variances = numeric_df.var()
print("--- Column Variances ---")
print(variances)

low_var_cols = variances[variances < 0.1].index.tolist()
df.drop(low_var_cols, axis=1, inplace=True)
print(f"\nDropped due to low variance: {low_var_cols}")

## 🛠️ Feature Engineering & Encoding

**Coach's Note:** Time to translate our scouting notes into machine tactics.

In [ ]:
# 1. Fix Wrong Formats: Convert 'Signing_Date' from string to datetime
df['Signing_Date'] = pd.to_datetime(df['Signing_Date'])
df['Signing_Year'] = df['Signing_Date'].dt.year
df.drop('Signing_Date', axis=1, inplace=True)

# 2. Split Features (X) and Labels (y)
X = df.drop('Goals_Scored', axis=1)
y = df['Goals_Scored']

# 3. Encode Categorical Columns
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce

# A. Ordinal Column (Medical Clearance: No < Yes)
le = LabelEncoder()
X['Medical_Clearance'] = le.fit_transform(X['Medical_Clearance'])

# B. Columns with 3-6 categories -> One-Hot (Position)
X = pd.get_dummies(X, columns=['Position'])

# C. High Cardinality Columns (If we had many clubs) -> Binary Encoding
# (Simulating a high cardinality case with dummy clubs)
X['Club'] = [f'Club_{np.random.randint(1, 50)}' for _ in range(len(X))]
be = ce.BinaryEncoder(cols=['Club'])
X = be.fit_transform(X)

print("\n--- Feature Engineering Complete ---")
X.head()